# Lab Stage 2: Processing Multi-Subject Datasets with High-Performance Computing

## Course architecture

Data acquisition ➡️ Training an ML model ➡️ Visualizing and judging results ➡️ Documenting research

## Instructions for the participant
In Lab Stage 1 you analyzed one participant on one machine. Here you train a machine learning model (a convolutional neural network, CNN) on EEG recordings from **all participants** in the dataset, to predict which of the four games each person was playing.

**Where FABRIC fits.** A laptop has about 8–16 CPU cores and 16–32 GB of RAM. FABRIC spans dozens of sites and adds GPU nodes; national resources such as [ACCESS](https://access-ci.org) and the [NAIRR Pilot](https://nairrpilot.org) go further still. Training on 28 people's EEG is where that jump starts to matter.

Your objective is to pass each **"AI Prompt Challenge"** to an AI coding assistant, check what it returns, and paste it into the matching code cell.

### ✅ Check the AI's code (every time)
Before you run any AI-generated code, ask three questions:
1. **Does it do what I asked?** Read it line by line against the prompt.
2. **Does it do anything I didn't ask for?** Downloads, file deletions, network calls, or changes to data it shouldn't touch.
3. **Can I explain each step in plain words?** If not, ask the AI to explain it before you run it.

In this lab one check matters most: **the three people held out to test the model must never appear in its training data.** A cell after training checks this for you.

***
## Module 0: Setting up an environment on FABRIC

**AI Prompt Challenge** — run the answer in your FABRIC node's **terminal**, not in a notebook cell:

> "I am working on a FABRIC HPC node running Ubuntu. Give me the terminal commands to (1) install Anaconda, (2) create a conda environment named 'eeg_hpc', (3) activate that environment, and (4) install numpy, pandas, matplotlib, and a GPU-compatible version of pytorch."

***
## Module 1: Data acquisition

We use the [EEG Emotion Recognition dataset](https://www.kaggle.com/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games) again, now for all 28 participants. The cell below downloads the full dataset from Kaggle.

In [ ]:
import kagglehub
import os

# Download from Kaggle
path = kagglehub.dataset_download("wajahat1064/emotion-recognition-using-eeg-and-computer-games")
print("Dataset downloaded to:", path)

path = os.path.join(path,
                    "Dataset - Emotion Recognition data Based on EEG Signals and Computer Games",
                    "Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO",
                    "GAMEEMO")

**AI Prompt Challenge**

The `subjects` variable should hold the participant identifiers used in the dataset, `"S01"` through `"S28"`. Design a prompt and define this variable with the AI.

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
print(len(subjects))

The functions below read one participant's four recordings and combine everyone into an array of shape (participants × games × channels × time).

In [ ]:
import numpy as np
import pandas as pd

def read_subject(subject, path):
    assert len(os.listdir(path)) == 4

    subject_recordings = []
    for game_idx in range(4):
        # Read time x channel data
        csv_filename = f"{subject}G{game_idx + 1}AllChannels.csv"
        df = pd.read_csv(os.path.join(path, csv_filename))
        df.drop("Unnamed: 14", axis=1, inplace=True)   # empty extra column at the end of each file

        # Shape into a channels x time array
        subject_recordings.append(df.to_numpy().T)

    # games x channels x time
    return np.stack(subject_recordings)

def read_subjects(subjects, path):
    all_recordings = []
    for subject in subjects:
        subject_path = os.path.join(path, f"({subject})", "Preprocessed EEG Data", ".csv format")
        all_recordings.append(read_subject(subject, subject_path))

    # participants x games x channels x time
    return np.stack(all_recordings)

Now try to read everyone. **This cell will stop with an error on one participant** — that is intended.

In [ ]:
data = read_subjects(subjects, path)

**AI Prompt Challenge**

There is an error when reading one participant's data. Design a prompt to list the contents of that participant's folder and compare them with another participant's (for example S01). What exactly is different?

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

**AI Prompt Challenge**

Prompt the AI to change the `subjects` list so that it excludes S26 but keeps S01–S25 and S27–S28.

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

In [ ]:
data = read_subjects(subjects, path)

# participants x games x channels x time
print(data.shape)

***
## Module 2: Training an ML model

We train a CNN to predict the game from the EEG recording. Each participant-and-game recording is one example (108 in total). **Three participants are held out** for testing: the model never trains on them, so their results show how it does on new people.

**AI Prompt Challenge**

Copy this text block into your AI assistant:

> "I am working with a dataset of shape subjects × conditions × channels × time; my actual shape is (27, 4, 14, 38252), stored in a numpy array called `data`. Write PyTorch code that trains a simple CNN to predict the condition (multiclass classification over 4 classes). Treat each subject × condition as a separate sample and make the CNN layers 1D (we convolve across time); the first layer should take 14 input channels. Use a fixed random seed to choose 3 subjects to hold out for validation and train on the other 24; store the subject indices in lists called `train_idx` and `val_idx`, and keep the validation inputs and labels as `X_val_t` (a tensor) and `y_val` (a numpy array). After every epoch, record training and validation loss and accuracy in lists called `train_losses`, `train_accs`, `val_losses` and `val_accs`. Keep the trained model in a variable called `model` and the device in `device`."

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

### ✅ Check the AI's code: are the held-out participants really held out?
Run this cell as given. It stops with an error if any held-out participant is also in the training set, or if the tracked lists are missing.

In [ ]:
print("Held-out participants:", [subjects[i] for i in val_idx])
assert not set(train_idx) & set(val_idx), "A held-out participant is also in the training set!"
assert len(train_idx) + len(val_idx) == len(subjects)
assert len(train_losses) == len(train_accs) == len(val_losses) == len(val_accs) > 0, "Tracked lists are missing or uneven"
print(f"OK: {len(train_idx)} training and {len(val_idx)} held-out participants, no overlap; "
      f"{len(val_accs)} epochs tracked.")

***
## Module 3: Visualizing and judging the results

**AI Prompt Challenge**

Design a prompt to plot the saved training and validation losses (one subplot) and accuracies (a second subplot) across epochs.

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

### How good is it, really?
With four games, random guessing is right **25%** of the time. The three held-out participants give only **12 test recordings**, so each one is worth about **8 percentage points** of accuracy: 33% is just one recording above chance.

**AI Prompt Challenge**

> "Using my trained PyTorch `model`, the held-out inputs `X_val_t` and labels `y_val`, compute the predicted game for each held-out recording. Print the test accuracy next to the chance level (25%), then build a 4×4 confusion matrix with numpy (rows = true game, columns = predicted game, labels boring, calm, horror, funny) and plot it with matplotlib, writing the count in each cell. Store the predictions in `y_pred` and the true labels in `y_true`."

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

**How often would guessing do this well?** Run this cell as given. It works out the chance that a model which only guesses would get at least as many of the 12 recordings right as yours did.

In [ ]:
from math import comb

n, k = len(y_true), int((y_pred == y_true).sum())
p_guess = sum(comb(n, i) * 0.25**i * 0.75**(n - i) for i in range(k, n + 1))
print(f"{k} of {n} correct. Pure guessing gets at least {k} right {p_guess:.0%} of the time.")

✍️ **Report honestly.** In one or two sentences below, state your test accuracy, the chance level, how often guessing would do as well, and which games the model confuses most. A result close to chance, reported honestly, is a valid finding.

*Your sentence here:*

***
## Module 4: Documenting research

**AI Prompt Challenge**

Design a prompt to save the per-epoch training and validation losses and accuracies to a `.csv` file, together with the held-out participants and the final test accuracy.

In [ ]:
# TODO: Paste your AI-generated code here, then run this cell.

***
## Module 5: Additional prompt challenges (stretch goals)

1. **Beat the Baseline:** Use AI to modify the CNN (layers, kernel sizes, channels, pooling) and try to improve held-out accuracy. Re-run the chance check: is the improvement bigger than one or two recordings?
2. **Try Different Optimizers:** Replace Adam with SGD, RMSprop, or AdamW and compare convergence speed and final accuracy.
3. **Learning Rate Tuning:** Add a learning rate scheduler (StepLR, CosineAnnealingLR, ReduceLROnPlateau) and see whether it helps.
4. **Normalization Challenge:** Z-score the EEG channels using statistics from the **training participants only**, and compare with the unnormalized data.
5. **Cross-Validation:** Repeat the training with 5-fold cross-validation across participants (about 5× the run time) and report the mean and spread of held-out accuracy. One split of three people is noisy; averaging over folds gives a more trustworthy estimate.